In [ ]:
import pandas as pd
import networkx as nx
import numpy as np

excel_file_path = './data/Menstrual_cramps.xlsx'

node_df = pd.read_excel(excel_file_path, sheet_name='경혈단독_count')

node_df['경혈명'] = node_df['경혈명'].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)

invalid_nodes = ['non-acupoint', 'ashi-point', 'BV', 'CV', 'FV']

valid_nodes = node_df[~node_df['경혈명'].isin(invalid_nodes)]['경혈명'].tolist()

edge_df = pd.read_excel(excel_file_path, sheet_name='경혈pair_count')

cols_to_clean = ['Source', 'Target']
for col in cols_to_clean:
    edge_df[col] = edge_df[col].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)

edge_df = edge_df[edge_df['Source'].isin(valid_nodes) & edge_df['Target'].isin(valid_nodes)]

G = nx.Graph()
G.add_nodes_from(valid_nodes) # 유효 노드만 추가
for _, row in edge_df.iterrows():
    G.add_edge(row['Source'], row['Target'], weight=float(row['Weight']))

print(f" 데이터 정제 및 그래프 생성 완료")
print(f" - 제외된 노드: {invalid_nodes}")
print(f" - 총 노드 수: {G.number_of_nodes()}개")
print(f" - 총 엣지 수: {G.number_of_edges()}개")

all_nodes_in_G = list(G.nodes())

kl_errors = [n for n in all_nodes_in_G if 'Kl' in n]
print(f"\n'Kl'(오타) 잔존 여부: {len(kl_errors)}개")

ki_check = [n for n in all_nodes_in_G if 'KI14' in n]
print(f"'KI14'(정상) 존재 여부: {ki_check}")

included_special = [n for n in all_nodes_in_G if n in invalid_nodes]
print(f"제외 대상 노드 잔존 여부: {len(included_special)}개")
if len(included_special) == 0:
    print("5개 노드가 그래프에서 모두 제거")
else:
    print(f"다음 노드들이 아직 남아있음: {included_special}")

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import random
import time

# 1. 확산 확률 설정 (현재의 G 기준)

current_nodes = list(G.nodes())

if G.number_of_edges() > 0:
    max_w = max(nx.get_edge_attributes(G, 'weight').values())
    edge_probs = {}
    for u, v, data in G.edges(data=True):
        p = (data['weight'] / max_w) ** 0.5 # Square Root Scaling
        edge_probs[(u, v)] = p
        edge_probs[(v, u)] = p
else:
    edge_probs = {}
    print("그래프에 엣지가 없습니다.")

# 2. Influence Maximization (사용자 정의 IC 모델)
def run_ic_simulation(graph, seeds, probabilities):
    active_nodes = set(seeds)
    newly_active = set(seeds)
    while newly_active:
        current_batch = list(newly_active)
        newly_active = set()
        for node in current_batch:
            neighbors = list(graph.neighbors(node))
            for neighbor in neighbors:
                if neighbor not in active_nodes:
                    prob = probabilities.get((node, neighbor), 0)
                    if random.random() < prob:
                        active_nodes.add(neighbor)
                        newly_active.add(neighbor)
    return active_nodes

def get_avg_spread(graph, seed_node, probabilities, sims=100):
    total_spread = 0
    for _ in range(sims):
        infected = run_ic_simulation(graph, [seed_node], probabilities)
        total_spread += len(infected)
    return total_spread / sims

SIMS = 1000
spread_results = {}

for i, node in enumerate(current_nodes):
    avg_spread = get_avg_spread(G, node, edge_probs, SIMS)
    spread_results[node] = avg_spread
    if (i+1) % 10 == 0:
        print(f"   > {i+1}/{len(current_nodes)} 완료...", end='\r')

# 상위 15개 추출 (경혈명, 전파력)
sorted_inf = sorted(spread_results.items(), key=lambda x: x[1], reverse=True)[:15]
top15_inf = [(node, round(score, 2)) for node, score in sorted_inf]

# 3. Centrality 지표 4종 계산
def get_top15_with_scores(centrality_dict):
    # 값 기준 내림차순 정렬 후 상위 15개 추출
    sorted_items = sorted(centrality_dict.items(), key=lambda x: x[1], reverse=True)[:15]
    return [(node, round(score, 4)) for node, score in sorted_items]

deg_cent = nx.degree_centrality(G)
top15_deg = get_top15_with_scores(deg_cent)

try:
    eig_cent = nx.eigenvector_centrality(G, max_iter=1000)
    top15_eig = get_top15_with_scores(eig_cent)
except:
    try:
        eig_cent = nx.eigenvector_centrality_numpy(G)
        top15_eig = get_top15_with_scores(eig_cent)
    except:
        top15_eig = [("Error", 0)] * 15

bet_cent = nx.betweenness_centrality(G)
top15_bet = get_top15_with_scores(bet_cent)

clo_cent = nx.closeness_centrality(G)
top15_clo = get_top15_with_scores(clo_cent)

# 4. 결과 통합 (Table 1 생성)
table_df = pd.DataFrame({
    'Rank': range(1, 16),
    'Influence Maximization': top15_inf,
    'Eigenvector Centrality': top15_eig,
    'Degree Centrality': top15_deg,
    'Betweenness Centrality': top15_bet,
    'Closeness Centrality': top15_clo
})

print("\n" + "="*100)
print(f"Table 1. 네트워크 영향력 지표 상위 15개 (제외 노드: {invalid_nodes})")
print("="*100)
print(table_df.to_string(index=False))